# Breast ultrasound: lesion classification and segmentation

Two models trained on breast ultrasound images:

1. a ResNet-34 classifier (normal / benign / malignant),
2. a U-Net with a ResNet-34 encoder that segments the lesion.

Training data is BUSI ([Al-Dhabyani et al., 2020](https://doi.org/10.1016/j.dib.2019.104863)): 780 images of 600 women aged 25 to 75, acquired in 2018 at Baheya Hospital in Cairo, each with a lesion mask. One thing to know about BUSI is that 155 of its images are copies of other images ([Pawłowska et al., 2023](https://doi.org/10.1016/j.dib.2023.109247)). If you split without removing them, some test images were already seen during training and the scores look better than they are. Section 3 removes them before the split.

The models are also tested on BUS-BRA ([Gómez-Flores et al., 2024](https://doi.org/10.1002/mp.16812)), 1,875 images of biopsy-proven lesions from 1,064 patients in Brazil. It is never used for training, only to see how the models do on data from another hospital.

Runs on Colab with a T4 GPU (*Runtime > Change runtime type*). It works on CPU too, just slower.

Student project, for research and teaching. Not meant for clinical use.

## 1. Setup

In [ ]:
%pip install -q kagglehub

In [ ]:
import hashlib
import random
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy import ndimage
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
                             classification_report, f1_score, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import tv_tensors
from torchvision.models import ResNet34_Weights, resnet34
from torchvision.transforms import v2 as T

In [ ]:
SEED = 42
IMG_SIZE = 256          # images are padded to a square, then resized (multiple of 32)
BATCH_SIZE = 16
EPOCHS_CLS = 30         # classifier
EPOCHS_SEG = 40         # U-Net
PATIENCE = 8            # stop when the validation score has not improved for this many epochs
LR = 3e-4
PRETRAINED = True       # start both networks from ImageNet weights
DUP_THRESHOLD = 0.9     # fingerprint correlation above which two images count as the same picture
MIN_LESION_PX = 50      # predicted blobs smaller than this many pixels are removed
RUN_EXTERNAL_TEST = True

BUSI_DIR = None         # folder of an already downloaded BUSI (with normal/, benign/, malignant/)
BUSBRA_DIR = None       # folder of an already downloaded BUS-BRA (with bus_data.csv)
DATA_DIR = Path("data")
MODEL_DIR = Path("models")
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

CLASSES = ["normal", "benign", "malignant"]
DEVICE = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
NUM_WORKERS = 2 if sys.platform.startswith("linux") else 0
USE_AMP = DEVICE.type == "cuda"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
print("PyTorch", torch.__version__, "| device:", DEVICE)

## 2. Data

BUSI is downloaded from Kaggle with `kagglehub`, no account needed. There is one folder per class, and the masks sit next to their image: `benign (12).png` has `benign (12)_mask.png`, plus `benign (12)_mask_1.png` when there is a second lesion. All the masks of an image are merged into one.

In [ ]:
def find_busi_root(path):
    # the Kaggle archive nests the class folders one level down
    path = Path(path)
    for folder in [path, *sorted(path.rglob("*"))]:
        if folder.is_dir() and all((folder / c).is_dir() for c in CLASSES):
            return folder
    raise FileNotFoundError(f"No normal/, benign/ and malignant/ folders under {path}")


if BUSI_DIR is None:
    import kagglehub
    busi_root = find_busi_root(kagglehub.dataset_download("aryashah2k/breast-ultrasound-images-dataset"))
else:
    busi_root = find_busi_root(BUSI_DIR)
print(busi_root)

In [ ]:
def index_busi(root):
    rows = []
    for label in CLASSES:
        files = sorted(p for p in (root / label).iterdir() if p.suffix.lower() == ".png")
        masks = {}
        for p in files:
            if "_mask" in p.stem:
                masks.setdefault(p.stem.split("_mask")[0], []).append(p)
        for p in files:
            if "_mask" not in p.stem:
                rows.append({"name": p.name, "label": label, "image": p, "masks": masks.get(p.stem, [])})
    df = pd.DataFrame(rows)
    df["y"] = df["label"].map(CLASSES.index)
    return df


df = index_busi(busi_root)
print(len(df), "images")
print(df["label"].value_counts().reindex(CLASSES).to_string())
print("images with several masks:", int((df["masks"].map(len) > 1).sum()))
print("images with no mask file:", int((df["masks"].map(len) == 0).sum()))

In [ ]:
def load_gray(path):
    return np.array(Image.open(path).convert("L"))


def pad_resize(arr, size, nearest=False):
    # pad to a square first so the resize does not distort the lesion
    h, w = arr.shape
    s = max(h, w)
    out = np.zeros((s, s), dtype=arr.dtype)
    top, left = (s - h) // 2, (s - w) // 2
    out[top:top + h, left:left + w] = arr
    resample = Image.Resampling.NEAREST if nearest else Image.Resampling.BILINEAR
    return np.array(Image.fromarray(out).resize((size, size), resample))


def load_case(image_path, mask_paths, size=IMG_SIZE):
    """Grayscale image (uint8) and lesion mask (bool), both size x size."""
    img = load_gray(image_path)
    mask = np.zeros(img.shape, dtype=np.uint8)
    for p in mask_paths:
        m = load_gray(p)
        if m.shape != img.shape:
            m = np.array(Image.fromarray(m).resize(img.shape[::-1], Image.Resampling.NEAREST))
        mask |= (m > 127).astype(np.uint8)
    return pad_resize(img, size), pad_resize(mask, size, nearest=True).astype(bool)


cases = [load_case(r.image, r.masks) for r in df.itertuples()]
images = np.stack([c[0] for c in cases])
masks = np.stack([c[1] for c in cases])
print(images.shape, masks.shape)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 9.5))
for row, label in zip(axes, CLASSES):
    for ax, i in zip(row, rng.choice(np.flatnonzero(df["label"] == label), 4, replace=False)):
        ax.imshow(images[i], cmap="gray")
        if masks[i].any():
            ax.contour(masks[i], levels=[0.5], colors="lime", linewidths=1)
        ax.set_title(df.at[i, "name"], fontsize=9)
        ax.axis("off")
fig.suptitle("BUSI samples, lesion masks in green")
plt.tight_layout()
plt.show()

## 3. Duplicates and split

To find duplicates I compare a fingerprint of each image. The image is shrunk to 128 × 128 and a blurred copy of itself is subtracted, which leaves the fine texture (speckle, edges). Then the mean fingerprint of the whole dataset is subtracted as well. Without that step, the black borders and the depth attenuation that all scans share make unrelated images look alike.

Two images whose fingerprints correlate above `DUP_THRESHOLD` are treated as the same picture. Some of these copies were saved with a different mask, a few even with a different label. Linked pairs are merged into groups, one image per group is kept, and groups whose labels disagree are dropped. This catches exact and near-exact copies, not crops or zoomed versions.

In [ ]:
def fingerprint(path, size=128):
    img = Image.open(path).convert("L").resize((size, size), Image.Resampling.BOX)
    x = torch.from_numpy(np.asarray(img, dtype=np.float32))[None, None]
    return (x - F.avg_pool2d(x, 9, stride=1, padding=4, count_include_pad=False)).flatten()


fps = torch.stack([fingerprint(p) for p in df["image"]])
fps = fps - fps.mean(0, keepdim=True)
fps = F.normalize(fps - fps.mean(1, keepdim=True), dim=1)
corr = (fps @ fps.T).numpy()
np.fill_diagonal(corr, -1)
pairs = np.argwhere(np.triu(corr, k=1) > DUP_THRESHOLD)

plt.figure(figsize=(6, 3))
plt.hist(corr.max(1), bins=60)
plt.axvline(DUP_THRESHOLD, color="red", ls="--", lw=1)
plt.xlabel("highest fingerprint correlation with another image")
plt.ylabel("images")
plt.tight_layout()
plt.show()

parent = np.arange(len(df))


def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


for i, j in pairs:
    parent[find(i)] = find(j)
df["group"] = [find(i) for i in range(len(df))]

in_group = df["group"].map(df["group"].value_counts()) > 1
conflict = df.groupby("group")["label"].transform("nunique") > 1
print(f"{len(pairs)} duplicate pairs: {int(in_group.sum())} images in {df.loc[in_group, 'group'].nunique()} groups")
print(f"{int(conflict.sum())} of them are in groups whose copies have different labels")

In [ ]:
if len(pairs):
    order = np.argsort(corr[pairs[:, 0], pairs[:, 1]])[:4]   # least similar pairs above the threshold
    fig, axes = plt.subplots(len(order), 2, figsize=(6.5, 3.2 * len(order)), squeeze=False)
    for (a, b), row in zip(pairs[order], axes):
        for ax, i in zip(row, (a, b)):
            ax.imshow(images[i], cmap="gray")
            if masks[i].any():
                ax.contour(masks[i], levels=[0.5], colors="lime", linewidths=1)
            ax.set_title(f"{df.at[i, 'label']}: {df.at[i, 'name']}", fontsize=9)
            ax.axis("off")
        row[0].text(1.03, -0.06, f"r = {corr[a, b]:.3f}", transform=row[0].transAxes, ha="center", fontsize=9)
    fig.suptitle("Least similar pairs above the threshold (should still be real copies)")
    plt.tight_layout()
    plt.show()

Stratified 70/15/15 split on what is left. Validation picks the best epoch, the test set is only used once at the end.

In [ ]:
clean = df[~conflict].drop_duplicates("group")
print(f"kept {len(clean)} of {len(df)} images")

tr, rest = train_test_split(clean.index, test_size=0.30, stratify=clean["y"], random_state=SEED)
va, te = train_test_split(rest, test_size=0.50, stratify=clean.loc[rest, "y"], random_state=SEED)
df["split"] = "dropped"
df.loc[tr, "split"] = "train"
df.loc[va, "split"] = "val"
df.loc[te, "split"] = "test"
df[["name", "label", "split"]].to_csv(DATA_DIR / "busi_split.csv", index=False)
pd.crosstab(df["label"], df["split"]).reindex(index=CLASSES, columns=["train", "val", "test", "dropped"], fill_value=0)

## 4. Datasets and augmentation

Image and mask get the same random flip, rotation, shift and zoom so they stay aligned. Brightness and contrast only change on the image. No vertical flip, the skin is always at the top of an ultrasound image. The grayscale image is repeated on 3 channels to match the ImageNet encoders.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
augment = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    T.ColorJitter(brightness=0.2, contrast=0.2),
])


def to_input(img):
    """uint8 image (H x W or 1 x H x W) to a normalized 3 x H x W float tensor."""
    x = torch.as_tensor(img).float().div(255)
    if x.ndim == 2:
        x = x[None]
    return (x.expand(3, -1, -1) - MEAN) / STD


class BreastUltrasound(Dataset):
    def __init__(self, images, masks, labels, train=False):
        self.images, self.masks, self.labels, self.train = images, masks, labels, train

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        img = tv_tensors.Image(torch.from_numpy(self.images[i])[None])
        mask = tv_tensors.Mask(torch.from_numpy(self.masks[i])[None].to(torch.uint8))
        if self.train:
            img, mask = augment(img, mask)
        return to_input(img.as_subclass(torch.Tensor)), mask.as_subclass(torch.Tensor).float(), int(self.labels[i])


def make_loader(split, train=False):
    idx = np.flatnonzero(df["split"] == split)
    ds = BreastUltrasound(images[idx], masks[idx], df["y"].to_numpy()[idx], train)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=train, drop_last=train,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")


train_loader, val_loader, test_loader = make_loader("train", train=True), make_loader("val"), make_loader("test")

In [ ]:
i = int(np.flatnonzero((df["split"] == "train") & (df["label"] == "malignant"))[0])
sample = BreastUltrasound(images[i:i + 1], masks[i:i + 1], [2], train=True)
fig, axes = plt.subplots(1, 5, figsize=(15, 3.4))
axes[0].imshow(images[i], cmap="gray")
axes[0].contour(masks[i], levels=[0.5], colors="lime", linewidths=1)
axes[0].set_title("original", fontsize=9)
for k, ax in enumerate(axes[1:], 1):
    x, m, _ = sample[0]
    ax.imshow((x * STD + MEAN)[0].clamp(0, 1), cmap="gray")
    ax.contour(m[0], levels=[0.5], colors="lime", linewidths=1)
    ax.set_title(f"augmented {k}", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

Both networks use the same training loop: AdamW, cosine schedule, mixed precision on GPU and early stopping. The weights of the best validation epoch are kept.

In [ ]:
def autocast():
    return torch.autocast(device_type="cuda" if USE_AMP else "cpu", enabled=USE_AMP)


def train_model(model, loss_fn, score_fn, epochs, name):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    best, best_state, wait, history, start = -np.inf, None, 0, [], time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        total, n = 0.0, 0
        for x, m, y in train_loader:
            x, m, y = x.to(DEVICE), m.to(DEVICE), y.to(DEVICE)
            with autocast():
                out = model(x)
            loss = loss_fn(out.float(), m, y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total, n = total + loss.item() * len(x), n + len(x)
        scheduler.step()
        score = score_fn(model, val_loader)
        history.append({"epoch": epoch, "train loss": total / n, "val score": score})
        print(f"epoch {epoch:2d}  train loss {total / n:.3f}  val score {score:.3f}")
        if score > best:
            best, wait = score, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:
                print("early stopping")
                break
    model.load_state_dict(best_state)
    torch.save(best_state, MODEL_DIR / f"{name}.pt")
    print(f"best val score {best:.3f}, {time.time() - start:.0f} s, saved to {MODEL_DIR / name}.pt")
    return pd.DataFrame(history)


def plot_history(history, score_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.2))
    ax1.plot(history["epoch"], history["train loss"])
    ax1.set(xlabel="epoch", title="training loss")
    ax2.plot(history["epoch"], history["val score"], color="tab:orange")
    ax2.set(xlabel="epoch", title=f"validation {score_name}")
    plt.tight_layout()
    plt.show()


def bootstrap_ci(metric, y, p, groups=None, n=1000):
    """95% CI over n resamples of the test set. With groups, whole groups (patients) are resampled."""
    r = np.random.default_rng(SEED)
    _, inv = np.unique(np.arange(len(y)) if groups is None else groups, return_inverse=True)
    members = [np.flatnonzero(inv == g) for g in range(inv.max() + 1)]
    vals = []
    for _ in range(n):
        i = np.concatenate([members[g] for g in r.integers(0, len(members), len(members))])
        if len(np.unique(y[i])) == len(np.unique(y)):   # every class still present
            vals.append(metric(y[i], p[i]))
    return np.percentile(vals, [2.5, 97.5])

## 5. Classification

ResNet-34 pretrained on ImageNet with a new 3-class head. More than half of BUSI is benign, so the cross-entropy is weighted by inverse class frequency to keep the normal and malignant classes from being ignored. The best epoch is picked on validation macro-F1.

In [ ]:
def resnet34_backbone():
    return resnet34(weights=ResNet34_Weights.IMAGENET1K_V1 if PRETRAINED else None)


classifier = resnet34_backbone()
classifier.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(classifier.fc.in_features, len(CLASSES)))
classifier = classifier.to(DEVICE)

train_y = df.loc[df["split"] == "train", "y"].to_numpy()
class_weights = len(train_y) / (len(CLASSES) * np.bincount(train_y, minlength=len(CLASSES)))
cross_entropy = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE),
                                    label_smoothing=0.05)


@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    probs, ys = [], []
    for x, _, y in loader:
        with autocast():
            probs.append(model(x.to(DEVICE)).float().softmax(1).cpu())
        ys.append(y)
    return torch.cat(probs).numpy(), torch.cat(ys).numpy()


def val_macro_f1(model, loader):
    p, y = predict_probs(model, loader)
    return f1_score(y, p.argmax(1), average="macro")


cls_history = train_model(classifier, lambda out, m, y: cross_entropy(out, y), val_macro_f1,
                          EPOCHS_CLS, "classifier_resnet34")
plot_history(cls_history, "macro-F1")

### Test results

Malignant sensitivity is the fraction of malignant images predicted malignant, and malignant specificity the fraction of the other images not predicted malignant. Confidence intervals come from 1,000 bootstrap resamples of the test set.

In [ ]:
p_test, y_test = predict_probs(classifier, test_loader)
pred_test = p_test.argmax(1)

cls_metrics = {
    "Accuracy": lambda y, p: accuracy_score(y, p.argmax(1)),
    "Balanced accuracy": lambda y, p: balanced_accuracy_score(y, p.argmax(1)),
    "Macro-F1": lambda y, p: f1_score(y, p.argmax(1), average="macro"),
    "ROC-AUC (one vs rest, macro)": lambda y, p: roc_auc_score(y, p, multi_class="ovr", average="macro"),
    "Malignant sensitivity": lambda y, p: np.mean(p.argmax(1)[y == 2] == 2),
    "Malignant specificity": lambda y, p: np.mean(p.argmax(1)[y != 2] != 2),
}
rows = []
for name, metric in cls_metrics.items():
    lo, hi = bootstrap_ci(metric, y_test, p_test)
    rows.append({"metric": name, "value": round(metric(y_test, p_test), 3), "95% CI": f"{lo:.3f} to {hi:.3f}"})
cls_results = pd.DataFrame(rows).set_index("metric")
print(f"{len(y_test)} test images")
cls_results

In [ ]:
print(classification_report(y_test, pred_test, target_names=CLASSES, digits=3, zero_division=0))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.6))
ConfusionMatrixDisplay.from_predictions(y_test, pred_test, display_labels=CLASSES, cmap="Blues",
                                        colorbar=False, ax=ax1)
ax1.set_title("Confusion matrix (test)")
for k, c in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(y_test == k, p_test[:, k])
    ax2.plot(fpr, tpr, label=f"{c} (AUC {roc_auc_score(y_test == k, p_test[:, k]):.3f})")
ax2.plot([0, 1], [0, 1], "k--", lw=0.8)
ax2.set(xlabel="false positive rate", ylabel="true positive rate", title="ROC, each class vs the rest (test)")
ax2.legend(loc="lower right")
plt.tight_layout()
plt.show()

### Grad-CAM

Grad-CAM ([Selvaraju et al., 2017](https://arxiv.org/abs/1610.02391)) shows which parts of the image the prediction depends on. Here it is mostly a sanity check: the heat should be on the lesion (green outline), not on the text, rulers or caliper marks that a lot of BUSI images have.

In [ ]:
def grad_cam(model, x, layer, cls=None):
    store = {}

    def hook(module, inputs, output):
        store["acts"] = output
        output.register_hook(lambda g: store.__setitem__("grads", g))

    handle = layer.register_forward_hook(hook)
    try:
        model.eval()
        logits = model(x[None].to(DEVICE))
        probs = logits.softmax(1)[0].detach().cpu().numpy()
        cls = int(probs.argmax()) if cls is None else cls
        model.zero_grad(set_to_none=True)
        logits[0, cls].backward()
    finally:
        handle.remove()
    weights = store["grads"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((weights * store["acts"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    return (cam / (cam.max() + 1e-8)).detach().cpu().numpy(), cls, probs


test_ids = np.flatnonzero(df["split"] == "test")
fig, axes = plt.subplots(2, 3, figsize=(12, 8.4))
for ax in axes.flat:
    ax.axis("off")
for col, k in zip(axes.T, range(len(CLASSES))):
    ids = test_ids[df["y"].to_numpy()[test_ids] == k]
    for ax, i in zip(col, rng.choice(ids, min(2, len(ids)), replace=False)):
        cam, cls, probs = grad_cam(classifier, to_input(images[i]), classifier.layer4)
        ax.imshow(images[i], cmap="gray")
        ax.imshow(cam, cmap="jet", alpha=0.35)
        if masks[i].any():
            ax.contour(masks[i], levels=[0.5], colors="lime", linewidths=1)
        ax.set_title(f"true {df.at[i, 'label']}, predicted {CLASSES[cls]} ({probs[cls]:.2f})", fontsize=9)
plt.tight_layout()
plt.show()

## 6. Segmentation

U-Net ([Ronneberger et al., 2015](https://arxiv.org/abs/1505.04597)) with the same pretrained ResNet-34 as encoder. It is trained on all images, normal ones included. Their mask is empty, so the network also learns when there is nothing to segment.

The loss is BCE + Dice. For validation I take the mean Dice over all images, where an empty prediction on a normal image counts as 1. Predicted blobs smaller than `MIN_LESION_PX` pixels are removed.

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        r = resnet34_backbone()
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)                    # 1/2, 64 channels
        self.pool = r.maxpool                                                  # 1/4
        self.enc1, self.enc2, self.enc3, self.enc4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.dec4 = DecoderBlock(512, 256, 256)                                # 1/16
        self.dec3 = DecoderBlock(256, 128, 128)                                # 1/8
        self.dec2 = DecoderBlock(128, 64, 64)                                  # 1/4
        self.dec1 = DecoderBlock(64, 64, 32)                                   # 1/2
        self.dec0 = DecoderBlock(32, 0, 16)                                    # full size
        self.head = nn.Conv2d(16, 1, 1)

    def forward(self, x):
        s = self.stem(x)
        e1 = self.enc1(self.pool(s))   # 1/4, 64 channels
        e2 = self.enc2(e1)             # 1/8, 128
        e3 = self.enc3(e2)             # 1/16, 256
        e4 = self.enc4(e3)             # 1/32, 512
        d = self.dec1(self.dec2(self.dec3(self.dec4(e4, e3), e2), e1), s)
        return self.head(self.dec0(d))


def seg_loss(logits, target):
    prob = torch.sigmoid(logits)
    inter = (prob * target).sum((1, 2, 3))
    dice = (2 * inter + 1) / (prob.sum((1, 2, 3)) + target.sum((1, 2, 3)) + 1)
    return F.binary_cross_entropy_with_logits(logits, target) + (1 - dice).mean()


def clean_mask(mask):
    """Remove predicted blobs smaller than MIN_LESION_PX pixels."""
    labels, n = ndimage.label(mask)
    if n == 0:
        return mask
    sizes = ndimage.sum(mask, labels, range(1, n + 1))
    return np.isin(labels, 1 + np.flatnonzero(sizes >= MIN_LESION_PX))


@torch.no_grad()
def predict_masks(model, loader):
    model.eval()
    out = []
    for x, _, _ in loader:
        with autocast():
            prob = torch.sigmoid(model(x.to(DEVICE)).float())[:, 0].cpu().numpy()
        out.extend(clean_mask(m) for m in prob > 0.5)
    return np.stack(out)


def dice_iou(pred, gt):
    inter, total = np.logical_and(pred, gt).sum(), pred.sum() + gt.sum()
    if total == 0:
        return 1.0, 1.0   # no lesion, none predicted
    return 2 * inter / total, inter / (total - inter)


def val_mean_dice(model, loader):
    pred = predict_masks(model, loader)
    return float(np.mean([dice_iou(p, g)[0] for p, g in zip(pred, loader.dataset.masks)]))


unet = UNet().to(DEVICE)
seg_history = train_model(unet, lambda out, m, y: seg_loss(out, m), val_mean_dice, EPOCHS_SEG, "unet_resnet34")
plot_history(seg_history, "mean Dice")

### Test results

Dice and IoU (1 = perfect overlap, 0 = none) are computed on the test images that have a lesion. For normal images the only question is whether the U-Net predicts a lesion anyway.

In [ ]:
pred_masks = predict_masks(unet, test_loader)
test_df = df[df["split"] == "test"].copy()
test_df["dice"], test_df["iou"] = zip(*[dice_iou(p, g) for p, g in zip(pred_masks, masks[test_df.index])])
test_df["predicted px"] = pred_masks.sum((1, 2))
lesion = (test_df["label"] != "normal").to_numpy()

rows = []
for name, sel in [("benign", test_df["label"] == "benign"), ("malignant", test_df["label"] == "malignant"),
                  ("all lesions", lesion)]:
    d = test_df.loc[sel, "dice"].to_numpy()
    lo, hi = bootstrap_ci(lambda y, p: p.mean(), test_df.loc[sel, "y"].to_numpy(), d)
    rows.append({"lesions": name, "images": len(d), "mean Dice": round(d.mean(), 3), "95% CI": f"{lo:.3f} to {hi:.3f}",
                 "mean IoU": round(test_df.loc[sel, "iou"].mean(), 3)})
seg_results = pd.DataFrame(rows).set_index("lesions")
false_alarms = int((test_df.loc[~lesion, "predicted px"] > 0).sum())
print(f"normal test images with a predicted lesion: {false_alarms} of {int((~lesion).sum())}")
seg_results

In [ ]:
ranked = test_df[lesion].sort_values("dice", ascending=False)
mid = len(ranked) // 2
shown = pd.concat([ranked.head(3), ranked.iloc[max(mid - 1, 0):mid + 2], ranked.tail(3)])
shown = shown[~shown.index.duplicated()]
fig, axes = plt.subplots(3, 3, figsize=(11, 11.4))
for ax in axes.flat:
    ax.axis("off")
for ax, (i, r) in zip(axes.flat, shown.iterrows()):
    k = test_df.index.get_loc(i)
    ax.imshow(images[i], cmap="gray")
    ax.contour(masks[i], levels=[0.5], colors="lime", linewidths=1.2)
    if pred_masks[k].any():
        ax.contour(pred_masks[k], levels=[0.5], colors="red", linewidths=1.2)
    ax.set_title(f"{r['label']}, Dice {r['dice']:.2f}", fontsize=9)
fig.suptitle("Best, median and worst test lesions (green: ground truth, red: prediction)")
plt.tight_layout()
plt.show()

## 7. Both models on one image

`analyze` takes a preprocessed image and returns the class probabilities, the Grad-CAM map and the predicted mask. For a scan of your own: `analyze(pad_resize(load_gray("scan.png"), IMG_SIZE))`.

In [ ]:
def analyze(img):
    x = to_input(img)
    cam, cls, probs = grad_cam(classifier, x, classifier.layer4)
    unet.eval()
    with torch.no_grad():
        prob = torch.sigmoid(unet(x[None].to(DEVICE)).float())[0, 0].cpu().numpy()
    return probs, cam, clean_mask(prob > 0.5)


fig, axes = plt.subplots(3, 3, figsize=(11, 11.4))
for row, k in zip(axes, range(len(CLASSES))):
    i = rng.choice(test_ids[df["y"].to_numpy()[test_ids] == k])
    probs, cam, pred = analyze(images[i])
    text = ", ".join(f"{c} {p:.2f}" for c, p in zip(CLASSES, probs))
    row[0].imshow(images[i], cmap="gray")
    if masks[i].any():
        row[0].contour(masks[i], levels=[0.5], colors="lime", linewidths=1)
    row[0].set_title(f"true: {CLASSES[k]}", fontsize=9)
    row[1].imshow(images[i], cmap="gray")
    row[1].imshow(cam, cmap="jet", alpha=0.35)
    row[1].set_title(text, fontsize=9)
    row[2].imshow(images[i], cmap="gray")
    if pred.any():
        row[2].contour(pred, levels=[0.5], colors="red", linewidths=1.2)
    row[2].set_title("predicted lesion" if pred.any() else "no lesion predicted", fontsize=9)
    for ax in row:
        ax.axis("off")
plt.tight_layout()
plt.show()

## 8. External test on BUS-BRA

BUS-BRA comes from another hospital, with other machines and other patients, and every lesion was confirmed by biopsy. There are no normal images, so the classifier is only evaluated on benign vs malignant, using its probability of malignant. The scores should drop compared to the BUSI test set, and how much they drop says how much the models depend on the hospital they were trained on. Many patients have several images, so the confidence interval resamples patients rather than images.

Downloaded from [Zenodo](https://doi.org/10.5281/zenodo.8231412) (134 MB, CC BY 4.0).

In [ ]:
BUSBRA_URL = "https://zenodo.org/records/8231412/files/BUSBRA.zip?download=1"
BUSBRA_MD5 = "1f8b2be6476d58fc97bfb5e5a1ea9bab"


def get_busbra():
    if BUSBRA_DIR is not None:
        return next(Path(BUSBRA_DIR).rglob("bus_data.csv")).parent
    target = DATA_DIR / "busbra"
    found = list(target.rglob("bus_data.csv")) if target.exists() else []
    if found:
        return found[0].parent
    zip_path = DATA_DIR / "BUSBRA.zip"
    if not zip_path.exists():
        print("downloading BUS-BRA from Zenodo...")
        urllib.request.urlretrieve(BUSBRA_URL, zip_path)
    md5 = hashlib.md5(zip_path.read_bytes()).hexdigest()
    if md5 != BUSBRA_MD5:
        raise ValueError(f"BUSBRA.zip has checksum {md5}, expected {BUSBRA_MD5}: delete it and run again")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(target)
    return next(target.rglob("bus_data.csv")).parent


busbra_root = None
if RUN_EXTERNAL_TEST:
    try:
        busbra_root = get_busbra()
        print(busbra_root)
    except Exception as e:
        print("external test skipped:", e)

In [ ]:
if busbra_root is not None:
    bb = pd.read_csv(busbra_root / "bus_data.csv")
    bb["y"] = np.where(bb["Pathology"].str.lower() == "malignant", 2, 1)
    bb_cases = [load_case(busbra_root / "Images" / f"{i}.png", [busbra_root / "Masks" / f"mask{i[3:]}.png"])
                for i in bb["ID"]]
    bb_images, bb_masks = np.stack([c[0] for c in bb_cases]), np.stack([c[1] for c in bb_cases])
    bb_loader = DataLoader(BreastUltrasound(bb_images, bb_masks, bb["y"].to_numpy()),
                           batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    print(len(bb), "images:", ", ".join(f"{n} {c}" for c, n in bb["Pathology"].str.lower().value_counts().items()))

    p_bb, y_bb = predict_probs(classifier, bb_loader)
    bb_pred = predict_masks(unet, bb_loader)
    bb_scores = np.array([dice_iou(p, g) for p, g in zip(bb_pred, bb_masks)])

    def lesion_scores(y, p, scores):
        malignant, pred = y == 2, p.argmax(1)
        return {"ROC-AUC, malignant vs benign": roc_auc_score(malignant, p[:, 2]),
                "malignant sensitivity": np.mean(pred[malignant] == 2),
                "benign specificity": np.mean(pred[~malignant] != 2),
                "lesions classified normal": np.mean(pred == 0),
                "mean Dice": scores[:, 0].mean(),
                "mean IoU": scores[:, 1].mean()}

    busi_scores = test_df.loc[lesion, ["dice", "iou"]].to_numpy()
    external = pd.DataFrame({
        f"BUSI test, {int(lesion.sum())} lesions": lesion_scores(y_test[lesion], p_test[lesion], busi_scores),
        f"BUS-BRA, {len(bb)} lesions": lesion_scores(y_bb, p_bb, bb_scores),
    }).round(3)
    lo, hi = bootstrap_ci(lambda y, p: roc_auc_score(y == 2, p[:, 2]), y_bb, p_bb, groups=bb["Case"].to_numpy())
    print(f"BUS-BRA ROC-AUC 95% CI (patients resampled): {lo:.3f} to {hi:.3f}")
    display(external)

In [ ]:
if busbra_root is not None:
    fig, axes = plt.subplots(2, 4, figsize=(14, 7.6))
    for ax, i in zip(axes.flat, rng.choice(len(bb), 8, replace=False)):
        ax.imshow(bb_images[i], cmap="gray")
        ax.contour(bb_masks[i], levels=[0.5], colors="lime", linewidths=1)
        if bb_pred[i].any():
            ax.contour(bb_pred[i], levels=[0.5], colors="red", linewidths=1)
        ax.set_title(f"{bb.at[i, 'Pathology']}, P(malignant) {p_bb[i, 2]:.2f}, Dice {bb_scores[i, 0]:.2f}", fontsize=8)
        ax.axis("off")
    fig.suptitle("BUS-BRA samples (green: ground truth, red: prediction)")
    plt.tight_layout()
    plt.show()

## 9. Summary

In [ ]:
print("Detection, BUSI test set")
display(cls_results)
print("Segmentation, BUSI test set")
display(seg_results)
print(f"normal test images with a predicted lesion: {false_alarms} of {int((~lesion).sum())}")
if busbra_root is not None:
    print("Lesions only, BUSI test set vs BUS-BRA")
    display(external)

### Limitations

- The test set is small (15% of the cleaned BUSI), hence the wide confidence intervals. A 5-fold cross-validation would give more stable numbers.
- BUSI has no patient IDs, so two different images of the same woman can end up one in train and one in test.
- All the training data comes from one hospital. Section 8 gives an idea of what happens elsewhere.
- The [2023 letter](https://doi.org/10.1016/j.dib.2023.109247) also lists at least 19 doubtful labels, 70 images of the armpit instead of the breast, and many images with text, measurements or calipers drawn on them. A CNN can learn those marks instead of the lesion, which is one reason to look at Grad-CAM. There is a [curated version of BUSI](https://doi.org/10.5281/zenodo.19047974) (Aumente-Maestro et al.) without the duplicates and problem images.
- None of this has been validated for clinical use.

### References

- W. Al-Dhabyani, M. Gomaa, H. Khaled, A. Fahmy. *Dataset of breast ultrasound images.* Data in Brief 28, 104863 (2020).
- A. Pawłowska, A. Karwat, N. Żołek. *Letter to the Editor. Re: "Dataset of breast ultrasound images".* Data in Brief 48, 109247 (2023).
- W. Gómez-Flores, M. J. Gregorio-Calas, W. C. de Albuquerque Pereira. *BUS-BRA: A breast ultrasound dataset for assessing computer-aided diagnosis systems.* Medical Physics 51(4), 3110-3123 (2024).
- K. He, X. Zhang, S. Ren, J. Sun. *Deep residual learning for image recognition.* CVPR (2016).
- O. Ronneberger, P. Fischer, T. Brox. *U-Net: Convolutional networks for biomedical image segmentation.* MICCAI (2015).
- R. R. Selvaraju et al. *Grad-CAM: Visual explanations from deep networks via gradient-based localization.* ICCV (2017).